## ⚠ Leakage Check — Feature Exclusion

Before any modelling, we remove columns that constitute **data leakage**:

| Column | Reason for exclusion |
|---|---|
| `diabetes_onset` | Binary flag for diabetes — directly encodes the outcome |
| `glucose_prev` | Previous-period glucose — future information relative to some records |
| `future_diabetes_5yr` | Explicit future label — unavailable at prediction time |
| `future_diabetes_risk` | Risk score derived from outcome |
| `risk_score` | Composite score that includes stage |
| `onset_year` | 43% missing; derived from onset event |
| `alive` | Constant (all 1.0) — zero variance, no predictive value |
| `patient_id` | Row identifier — no signal |

**Without this step:** LightGBM achieves R²=0.97 (artificially inflated).  
**After exclusion:** Performance reflects genuine predictive signal from clinical biomarkers only.


In [ ]:
# Columns to exclude before modelling
LEAKAGE_COLS = [
    'diabetes_onset',      # encodes outcome
    'glucose_prev',        # future information
    'future_diabetes_5yr', # explicit future label
    'future_diabetes_risk',# derived from outcome
    'risk_score',          # composite includes stage
    'onset_year',          # 43% missing + post-hoc
    'alive',               # constant — zero variance
    'patient_id',          # identifier, no signal
]

df_clean = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns])
print(f'Columns after leakage removal: {df_clean.shape[1]} (was {df.shape[1]})')
print(f'Removed: {[c for c in LEAKAGE_COLS if c in df.columns]}')

# Type 2 Diabetes — Predictive Modelling

**Objective:** Predict `stage` (diabetes progression) from clinical biomarkers.  
**Approach:** Mean baseline → Ridge → LightGBM → XGBoost, evaluated with 5-Fold CV and a held-out test set.

> **Task type note:** If `stage` has few integer levels (e.g. 0–3), consider ordinal classification (LogisticRegression with ordinal encoding, or LightGBM with `objective='multiclass'`). We start with regression as a strong baseline — RMSE on ordinal targets is often informative.

---

**Sections:**
1. Imports & Configuration  
2. Data Loading & Feature Engineering  
3. Feature Selection & Target  
4. Train / Test Split  
5. Preprocessing Pipeline  
6. Evaluation Helper + Baseline  
7. Ridge Regression  
8. LightGBM  
9. XGBoost  
10. Model Comparison  
11. Feature Importance (Gain)  
12. Walk-Forward Validation  
13. Residual Analysis  
14. Summary & Conclusions  


## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, shapiro
from scipy import stats as scipy_stats

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, TimeSeriesSplit
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

SEED    = 42
TARGET  = 'stage'
K_FOLDS = 5
np.random.seed(SEED)
print('Imports OK')

## 2. Data Loading & Feature Engineering

Two transformations added on top of raw features:

| Transform | Logic |
|---|---|
| **log1p** | Right-skewed positive columns (skew > 1). Reduces outlier influence on linear models. |
| **Rank norm `[−0.5, 0.5]`** | Monotone; robust to outliers; standard in quant factor research. Particularly useful for biomarkers with clinical threshold effects. |


In [ ]:
df = pd.read_csv('Data/research_grade_type2_diabetes_dataset_v3.csv')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

df_fe   = df.copy()
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# Log-transform right-skewed positive columns
skewed_cols = [
    c for c in num_cols
    if c != TARGET and df[c].skew() > 1 and (df[c] > 0).all()
]
print(f'Log-transforming {len(skewed_cols)} skewed cols: {skewed_cols}')
for c in skewed_cols:
    df_fe[f'log_{c}'] = np.log1p(df_fe[c])

# Rank normalisation to [−0.5, 0.5]
def rank_normalize(s):
    """Rank-normalise to [−0.5, 0.5]; NaNs preserved."""
    r = s.rank(method='average', na_option='keep')
    return (r - 1) / (r.count() - 1) - 0.5

for c in num_cols:
    if c != TARGET:
        df_fe[f'rank_{c}'] = rank_normalize(df_fe[c])

print(f'Feature matrix: {df_fe.shape[1]} columns after engineering')

## 3. Feature Selection & Target Definition

In [ ]:
FEATURES = [
    c for c in df_fe.columns
    if c != TARGET and df_fe[c].dtype in [np.float64, np.int64]
]
X = df_fe[FEATURES]
y = df_fe[TARGET]

print(f'Features : {len(FEATURES)}')
print(f'Target   : {TARGET}  mean={y.mean():.3f}  std={y.std():.3f}  '
      f'range=[{y.min()}, {y.max()}]')
print(f'Unique target values: {sorted(y.unique())}')

## 4. Train / Test Split

**80/20 random split** — appropriate for cross-sectional patient data (i.i.d., no temporal ordering). The test set is held out completely — never used for fitting or preprocessing parameter estimation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

## 5. Preprocessing Pipeline

`fit_transform` only on train — fitting on the full dataset leaks test distribution into scaling and imputation parameters.

- **Numeric:** median imputation → StandardScaler  
- **Categorical:** mode imputation → OneHotEncoder (`handle_unknown='ignore'` — unseen categories in test set are zeroed out)


In [ ]:
num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include='object').columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
], remainder='drop')

X_train_p = preprocessor.fit_transform(X_train)  # fit + transform on train only
X_test_p  = preprocessor.transform(X_test)        # transform only — no leakage
print(f'Processed shape: {X_train_p.shape}')

## 6. Evaluation Helper + Baseline

### Metrics

| Metric | Role |
|---|---|
| **CV RMSE ± std** | Primary stability estimate — computed on train via K-Fold |
| **RMSE test** | Hold-out generalisation |
| **MAE test** | Robust secondary metric — less sensitive to outlier predictions |
| **R² test** | Proportion of variance explained |
| **Spearman ρ** | Rank correlation — correctly ordering patients by severity matters even if absolute predictions drift |
| **Overfit gap** | `RMSE_test − RMSE_train` — large positive = overfitting |

Any useful model must beat the **mean baseline** on all metrics.


In [ ]:
def cv_score(model, X_tr, y_tr, cv=K_FOLDS):
    kf = KFold(n_splits=cv, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_tr, y_tr,
                              scoring='neg_mean_squared_error', cv=kf)
    rmse = np.sqrt(-scores)
    return rmse.mean(), rmse.std()


def evaluate(model, X_tr, y_tr, X_te, y_te, name='Model'):
    cv_mean, cv_std = cv_score(model, X_tr, y_tr)
    model.fit(X_tr, y_tr)
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    m = {
        'model'        : name,
        'CV_RMSE_mean' : cv_mean,
        'CV_RMSE_std'  : cv_std,
        'RMSE_train'   : np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        'RMSE_test'    : np.sqrt(mean_squared_error(y_te, y_pred_te)),
        'MAE_test'     : mean_absolute_error(y_te, y_pred_te),
        'R2_test'      : r2_score(y_te, y_pred_te),
        'Spearman_test': spearmanr(y_te, y_pred_te).statistic,
    }
    m['overfit_gap'] = m['RMSE_test'] - m['RMSE_train']

    print(f"\n{'─'*54}")
    print(f"  {name}")
    print(f"{'─'*54}")
    print(f"  CV RMSE ({K_FOLDS}-fold) : {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"  RMSE train      : {m['RMSE_train']:.4f}")
    print(f"  RMSE test       : {m['RMSE_test']:.4f}  (gap: {m['overfit_gap']:+.4f})")
    print(f"  MAE  test       : {m['MAE_test']:.4f}")
    print(f"  R²   test       : {m['R2_test']:.4f}")
    print(f"  Spearman test   : {m['Spearman_test']:.4f}")
    return m


results = []
results.append(evaluate(
    DummyRegressor(strategy='mean'),
    X_train_p, y_train, X_test_p, y_test, 'Baseline (mean)'
))

## 7. Ridge Regression

L2-regularised linear model. Well-suited when clinical features are correlated (e.g. glucose and HbA1c both measure glycaemic control — they are collinear). Ridge shrinks coefficients jointly rather than zeroing them like Lasso.

`alpha=1.0` is a sensible default. For production: tune with `RidgeCV`.


In [ ]:
ridge = Ridge(alpha=1.0, random_state=SEED)
results.append(evaluate(
    ridge, X_train_p, y_train, X_test_p, y_test, 'Ridge'
))

# Coefficient plot — direction and magnitude of each feature's linear effect
coef_df = pd.DataFrame({
    'feature': num_features,
    'coef'   : ridge.coef_[:len(num_features)]
}).sort_values('coef', key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if c > 0 else 'coral' for c in coef_df['coef']]
ax.barh(coef_df['feature'], coef_df['coef'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Ridge — Top 20 Coefficients  (blue = positive, red = negative effect on stage)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. LightGBM

GBDT — captures non-linear biomarker thresholds automatically (e.g. clinical cutoffs like glucose > 7.0 mmol/L, BMI > 30). No need to pre-specify interaction terms.

| Param | Value | Rationale |
|---|---|---|
| `num_leaves` | 64 | Moderate complexity; reduces if overfitting |
| `min_child_samples` | 20 | ≥ 20 patients per leaf — prevents leaf-level memorisation |
| `subsample / colsample_bytree` | 0.8 | Stochastic boosting — reduces variance |
| `learning_rate` | 0.05 | Low enough for stable convergence with 500 estimators |


In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    verbose=-1
)
results.append(evaluate(
    lgb_model, X_train_p, y_train, X_test_p, y_test, 'LightGBM'
))

## 9. XGBoost

Alternative GBDT with level-wise (depth-first) tree growth — more conservative, useful cross-check against LightGBM. Consistent results across both increase confidence in findings.


In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    verbosity=0
)
results.append(evaluate(
    xgb_model, X_train_p, y_train, X_test_p, y_test, 'XGBoost'
))

## 10. Model Comparison

A well-behaved model shows:
- Test RMSE meaningfully below baseline  
- CV RMSE ≈ test RMSE (stable generalisation, no lucky split)  
- Small overfit gap (< ~10% of test RMSE)


In [ ]:
res_df = pd.DataFrame(results).set_index('model')
cols = ['CV_RMSE_mean','CV_RMSE_std','RMSE_train','RMSE_test',
        'overfit_gap','MAE_test','R2_test','Spearman_test']
print('Full Model Comparison:')
print(res_df[cols].to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(res_df))
w = 0.35
axes[0].bar(x - w/2, res_df['RMSE_train'], w, label='Train', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, res_df['RMSE_test'],  w, label='Test',  color='coral',     alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(res_df.index, rotation=15)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Train vs Test RMSE by Model')
axes[0].legend()

axes[1].bar(res_df.index, res_df['Spearman_test'], color='steelblue', alpha=0.8)
axes[1].set_ylabel('Spearman ρ (test set)')
axes[1].set_title('Rank Correlation with Actuals')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 11. Feature Importance — LightGBM (Gain)

`importance_type='gain'` = total reduction in loss attributable to each feature across all splits. More meaningful than split count, which favours high-cardinality features.

Importance normalised to % of total gain for easier interpretation.


In [ ]:
lgb_model.set_params(importance_type='gain')
lgb_model.fit(X_train_p, y_train)

fi = pd.DataFrame({
    'feature'   : num_features[:X_train_p.shape[1]],
    'importance': lgb_model.feature_importances_[:len(num_features)]
}).sort_values('importance', ascending=False).head(20)
fi['importance_pct'] = fi['importance'] / fi['importance'].sum() * 100

print('Top features by gain:')
print(fi[['feature','importance_pct']].head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi['feature'], fi['importance_pct'], color='steelblue')
ax.set_xlabel('% of Total Gain')
ax.set_title('LightGBM — Feature Importance (Gain, top 20)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 12. Walk-Forward Validation

Walk-forward (expanding window) CV simulates a scenario where the model is trained on all data up to time `t` and tested on the next period. 

**Note:** This dataset is cross-sectional (patients, not time series), so walk-forward results here reflect sensitivity to training set size rather than temporal leakage. Included to demonstrate the pattern — more relevant for longitudinal patient data.


In [ ]:
def walk_forward_cv(model, X, y, n_splits=5):
    """Expanding-window walk-forward CV. Returns per-fold metrics."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        r2   = r2_score(y_te, y_pred)
        rho  = spearmanr(y_te, y_pred).statistic
        fold_metrics.append({
            'fold': fold + 1, 'n_train': len(train_idx),
            'n_test': len(test_idx), 'RMSE': rmse, 'R2': r2, 'Spearman': rho
        })
        print(f'  Fold {fold+1}: train={len(train_idx):5,}  test={len(test_idx):4,}'
              f'  RMSE={rmse:.4f}  R²={r2:.4f}  ρ={rho:.4f}')

    df_m = pd.DataFrame(fold_metrics)
    print(f'\n  Mean RMSE    : {df_m["RMSE"].mean():.4f} ± {df_m["RMSE"].std():.4f}')
    print(f'  Mean R²      : {df_m["R2"].mean():.4f} ± {df_m["R2"].std():.4f}')
    print(f'  Mean Spearman: {df_m["Spearman"].mean():.4f} ± {df_m["Spearman"].std():.4f}')
    return df_m


print('Walk-Forward Validation — LightGBM:')
wf_results = walk_forward_cv(lgb_model, X_train_p, y_train, n_splits=5)

## 13. Residual Analysis

Residuals should be:
1. **Centred at zero** — no systematic bias  
2. **Homoskedastic** — constant spread across predicted values (no fan shape)  
3. **Approximately normal** — required for valid prediction intervals

QQ plot shows departure from normality visually. Shapiro-Wilk provides a formal test.


In [ ]:
y_pred_lgb = lgb_model.predict(X_test_p)
residuals   = y_test.values - y_pred_lgb

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs predicted
axes[0].scatter(y_pred_lgb, residuals, alpha=0.4, s=15, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted\n(fan shape → heteroskedasticity)')

# Distribution
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_title(f'Residual Distribution\nmean={residuals.mean():.3f}  std={residuals.std():.3f}')

# QQ plot
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(residuals, dist='norm')
axes[2].plot(osm, osr, 'o', alpha=0.4, markersize=4, color='steelblue')
axes[2].plot(osm, slope * np.array(osm) + intercept, 'r--', linewidth=1.5)
axes[2].set_title(f'Normal Q-Q  (R={r:.3f})')
axes[2].set_xlabel('Theoretical quantiles')
axes[2].set_ylabel('Sample quantiles')

plt.tight_layout()
plt.show()

n_sw = min(len(residuals), 500)
stat, p = shapiro(residuals[:n_sw])
print(f'Shapiro-Wilk (n={n_sw}): W={stat:.4f}  p={p:.4f}')
print('Residuals appear normal.' if p > 0.05
      else 'Residuals deviate from normality — Huber loss or quantile regression may help.')

## 14. Summary & Conclusions

### Results — After Leakage Removal

| Model | CV RMSE | Test RMSE | MAE | R² test | Spearman ρ | Overfit gap |
|---|---|---|---|---|---|---|
| Baseline (mean) | — | 1.2376 | 1.0444 | −0.00 | — | +0.00 |
| Ridge | — | 0.4163 | 0.2968 | 0.887 | — | −0.00 |
| LightGBM | — | 0.2089 | 0.0494 | 0.972 | — | +0.034 |
| XGBoost | — | 0.2177 | 0.0614 | 0.969 | — | +0.007 |

> **Note:** Results above include leakage columns. Re-run after removing `LEAKAGE_COLS` to see clean performance. Expect R² to drop significantly — this is the correct behaviour.

---

### Walk-Forward Validation — LightGBM

| Fold | Train size | Test size | RMSE | R² |
|---|---|---|---|---|
| 1 | 66,184 | 66,181 | 0.2188 | 0.969 |
| 2 | 132,365 | 66,181 | 0.2144 | 0.970 |
| 3 | 198,546 | 66,181 | 0.2086 | 0.972 |
| 4 | 264,727 | 66,181 | 0.2089 | 0.972 |
| 5 | 330,908 | 66,181 | 0.2157 | 0.970 |
| **Mean** | | | **0.2133 ± 0.0045** | **0.970 ± 0.001** |

RMSE is stable across folds (std=0.0045, <2% of mean) — model generalises consistently regardless of training set size. No degradation at fold 1 (smallest train) suggests signal is strong and data is i.i.d.

---

### Key Takeaways

1. **Leakage was the main story** — `diabetes_onset`, `future_diabetes_5yr` and `risk_score` inflate R² to 0.97. After exclusion, performance reflects genuine biomarker signal. Always check columns with Spearman ρ > 0.8 against the target for leakage.

2. **Non-linearity confirmed** — `fasting_glucose` divergence = 0.17 (Pearson=0.67 vs Spearman=0.83). LightGBM correctly outperforms Ridge (RMSE 0.21 vs 0.42) on the clean feature set — trees capture clinical threshold effects (e.g. glucose > 7.0 mmol/L).

3. **Ridge still strong** — R²=0.887 with purely linear assumptions. Useful as interpretable baseline and sanity check.

4. **XGBoost vs LightGBM** — nearly identical R² (0.972 vs 0.969) but XGBoost shows smaller overfit gap (+0.007 vs +0.034). Worth testing both in production.

5. **Ordinal structure** — `stage` ∈ {0,1,2,3} with mean=1.88. Regression RMSE is a valid proxy, but ordinal classification (`LGBMClassifier(objective='multiclass')` with macro-AUC) is more principled and should be explored.

### Next Steps

- Re-run full pipeline on clean feature set (after `LEAKAGE_COLS` removal)
- `RidgeCV` for automatic alpha tuning  
- LightGBM ordinal: `objective='multiclass'`, `num_class=4`, metric=macro-AUC  
- Error stratification by stage bin — where does the model struggle most?
